# 03 — Génération d'idées par RAG (Sprint D)

Démo du pipeline : corpus → retrieval → génération (LLM ou mock) → scoring → classement.
Réutilise `src/generator/`.

Sans `LLM_API_KEY` dans `.env`, la génération bascule en **mock** (le scoring reste réel).

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from dotenv import load_dotenv
load_dotenv('../.env')

from src.generator import corpus
from src.generator.retriever import Retriever
from src.generator.generate import generate_ideas

## 1. Corpus documentaire (grounding)

In [2]:
docs = corpus.load_documents()
print('Documents :', corpus.stats(docs))

Documents : {'total': 997, 'by_source': {'style': 476, 'keyword': 350, 'trend': 171}}


## 2. Contexte récupéré pour une niche (retrieval)

In [3]:
retr = Retriever(docs)
ctx = retr.build_context('QUIZ_FOOT', 'FR')
print('Exemples à succès :', ctx['examples'][:3])
print('Tendances         :', ctx['trends'][:5])
print('Mots-clés         :', ctx['keywords'][:8])

Exemples à succès : ['DEVINE QUI EST LE JOUEUR PRO ! (Ft. SDM et Alonz)', 'Devine le FOOTBALLEUR avec les CHEVEUX, le CLUB, la CHANSON | Trouve Ronaldo, Mbappé, Messi, Neymar', 'Devine le vrai LOGO...! ⚽✅ | Quiz Logo Football 2024 👀']
Tendances         : ['quiz football joueur', 'quiz football joueur', 'quiz football', 'quiz foot difficile', 'quiz foot facile']
Mots-clés         : ['football', 'foot', 'devine', 'joueur', 'devine joueur', 'tes', 'coupe', 'monde']


## 3. Génération + scoring + classement

`mock` est déterminé automatiquement : `True` si aucune clé LLM.

In [4]:
client = None
try:
    from src.generator.llm import LLMClient
    client = LLMClient() if os.getenv('LLM_API_KEY') else None
except Exception:
    client = None
print('LLM réel :', client is not None)

ideas = generate_ideas('QUIZ_FOOT', 'FR', n=5, retriever=retr, client=client, mock=client is None)
pd.DataFrame(ideas)[['predicted_score', 'family', 'title']]

LLM réel : True


,predicted_score,family,title
0,0.144,LETTRES_MANQUANTES,LETTRES MANQUANTES : Les JOUEURS CÉLÈBRES
1,-0.094,REBUS_EMOJIS,EMOJIS : Devine le FOOTBALLEUR !
2,-0.192,VRAI_FAUX,VRAI ou FAUX : Faits Insolites sur le FOOT
3,-0.415,MOT_MELANGE,MOT MÉLANGÉ : Clubs de FOOT !
4,-1.018,MCQ,Devine le JOUEUR avec ses STATISTIQUES !


## 4. Conclusions

- Le RAG ancre la génération sur des **titres à succès + tendances + mots-clés** réels de la niche.
- Chaque idée est **re-scorée** par le modèle du notebook 02 et **classée** par potentiel viral.
- La même logique est exposée par l'**API** (`POST /generate`) et la **démo Streamlit**.